# Track 10 — Capstone: Production Harness (eval·CI·capstone_runner 통합)

## 운영 하네스란?

캡스톤을 한 번 실행하는 것과, 같은 결과를 반복 검증하고 CI로 지키는 것은 다릅니다. 운영 하네스는 그 반복 가능한 경로를 묶습니다.

- **`capstone_runner`:** 캡스톤 회귀를 노트북 밖에서 실행하고 리포트를 남깁니다.
- **정적 회귀:** 골든셋으로 M1/M6/M9를 채점합니다.
- **라이브 스모크:** API 키가 있으면 `eval.run`으로 실제 모델·도구 경로를 확인합니다.
- **CI 템플릿:** 매 PR마다 정적 회귀를 다시 실행할 진입점을 제공합니다.
- **운영 패키지:** 위 결과와 SLO를 `capstone_package.json`으로 묶습니다.

## 이 노트북에서 보여줄 것

| Session | 보여주는 것 | 목적 |
|---|---|---|
| 1. Setup | facade 시작 · 골든 로더 · 정적 회귀 헬퍼 · 패키지 저장 헬퍼 | 공통 준비 |
| 2. ① capstone_runner | 공통 골든 22행 채점과 `runner_report.json` 수집 | 정적 회귀 기준선 |
| 3. ② eval.run 스모크 | 키가 있으면 BFCL 1건 라이브 실행, 없으면 건너뜀 | 라이브 경로 확인 |
| 4. ③ CI workflow 템플릿 | 회귀 workflow 복사와 트리거 확인 | PR 회귀 게이트 준비 |
| 5. ④ 패키지 | ①~③ 결과와 정적 회귀를 한 JSON으로 저장 | 운영 산출물 마감 |

## 이 노트북을 마치면

- `capstone_runner`, 정적 회귀, 라이브 스모크, CI 템플릿을 하나의 운영 파이프라인으로 연결할 수 있습니다.
- 정적 fixture 회귀와 라이브 eval 스모크를 구분할 수 있습니다.
- CLI 경로와 노트북 경로가 같은 골든셋을 보는지 확인할 수 있습니다.

**산출물:** `_out/07/capstone_package.json`  
**실행 조건:** Session 2·4·5는 API 키 없이 실행됩니다. Session 3 라이브 스모크만 API 키가 필요합니다. 정적 단계는 `eval/` 패키지가 import 경로에 있어야 합니다.

> 요약: capstone_runner·정적 회귀·라이브 스모크·CI 템플릿을 운영 패키지로 묶는 캡스톤입니다.


## 파이프라인 한눈에 보기

골든셋과 라이브 스모크를 따로 채점한 뒤, 결과를 하나의 운영 패키지로 모읍니다. 박스 ①~④는 Session 2~5와 대응하며, 코드 셀의 `STEP n/4` 배너가 같은 단계를 가리킵니다.

```mermaid
flowchart LR
    G[("골든셋<br/>'all' 22행")] --> S1["① capstone_runner<br/>정적 회귀 채점<br/>(Session 2)"]
    BF[("BFCL 1건")] --> S2["② eval.run 스모크<br/>라이브 모델·도구 호출<br/>(Session 3)"]
    S1 -.->|매 PR마다 ① 재실행| S3["③ CI 회귀 게이트<br/>(Session 4)"]
    S1 --> S4["④ 운영 패키지<br/>SLO+회귀+스모크+CI<br/>(Session 5)"]
    S2 --> S4
    S3 --> S4
    S4 --> P[("capstone_package.json")]
```

| 단계 | Session | 입력 → 동작 → 산출물 |
|---|---|---|
| ① 정적 회귀 | 2 | 골든 22행 → `capstone_runner.py` 채점 → `runner_report.json` |
| ② 라이브 스모크 | 3 | BFCL 1건 → 실제 모델·도구 호출 → `eval/reports/*.json` |
| ③ CI 게이트 | 4 | yml 템플릿 → PR마다 ① 재실행 → `capstone_regression.yml` |
| ④ 패키지 | 5 | ①·②·③ 결과 → SLO와 묶기 → `capstone_package.json` |

> 핵심: ①·④는 미리 적힌 골든 텍스트를 채점하는 정적 경로이고, ②는 별도 BFCL 입력으로 실제 모델 호출을 실행하는 라이브 경로입니다.


## Session 1. Setup


### Session 1-1. Setup

**하는 일:** 경로(절대경로) · 골든 로더 · 정적 회귀 헬퍼 · 패키지 저장 헬퍼를 준비합니다.

**정상:** `exaone 0.1.0 | HAS_API True/False` 한 줄이 출력됩니다.

**의미:** 이후 네 단계가 공유할 경로·헬퍼가 준비됐는지 먼저 봅니다.

In [ ]:
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import logging

# (en) Quiet library logs so the notebook output stays readable.
# (kr) 라이브러리 로그를 줄여 노트북 출력을 읽기 쉽게 한다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)

# (en) Facade-only startup; requires editable install at the repo root.
# (kr) `exaone` facade로 시작한다. 저장소 루트에서 editable 설치가 필요하다.
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "`exaone`이 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
ROOT = exaone.project_root()
TRACK10 = ROOT / "recipes" / "track10_ax_capstones"
DATA = TRACK10 / "data"
SNIPPETS_PATH = ROOT / "recipes" / "track04_rag_and_knowledge" / "data" / "internal_manual_snippets.json"
API_KEY = os.environ.get("EXAONE_API_KEY", "").strip()
BASE_URL = os.environ.get("EXAONE_BASE_URL", "").strip() or "http://localhost:8000/v1"
MODEL = os.environ.get("EXAONE_MODEL", "").strip() or exaone.llm.ExaoneClient.DEFAULT_MODEL
HAS_API = bool(API_KEY)
client = None
if HAS_API:
    client = exaone.llm.ExaoneAPIClient(base_url=BASE_URL, model=MODEL, api_key=API_KEY)
print("exaone", exaone.__version__, "| HAS_API", HAS_API)

def load_capstone_golden(tag: str) -> list[dict]:
    # (en) Load golden rows for this capstone id or shared "all" rows.
    # (kr) 해당 캡스톤과 공통 "all" 골든 사례를 함께 불러온다.
    rows: list[dict] = []
    for line in (DATA / "capstone_golden.jsonl").read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        if row.get("capstone") in (tag, "all"):
            rows.append(row)
    return rows


def regression_m1_m6_m9(rows: list[dict]) -> dict:
    # (en) Static fixture metric demo (M1/M6/M9) over golden rows — metric MECHANICS, not live agent.
    # (kr) 정적 골든 fixture로 M1/M6/M9를 계산한다. 라이브 에이전트 성능이 아니라 메트릭 동작 예시다.
    from eval.metrics.m1_task_success import TaskGold
    from eval.metrics.m6_schema_adherence import SchemaSpec
    from eval.metrics.m9_faithfulness import LengthRatioJudge
    from eval.metrics import m1_task_success, m6_schema_adherence
    from eval.metrics.types import TrialResult

    m1s, m6s, m9s = [], [], []
    cases = []
    for row in rows:
        tid = row["id"]
        content = row.get("trial_content") or str(row.get("expected_answer", ""))
        tr = TrialResult(
            trial_id=f"cap-{tid}",
            task_id=tid,
            dataset="track10.golden",
            runner="capstone",
            final_content=content,
        )
        m1 = m6 = m9 = None
        if row.get("expected_answer") is not None:
            m1 = m1_task_success.score_trial_exact(tr, TaskGold(task_id=tid, answer=row["expected_answer"]))
            m1s.append(m1)
        rk = row.get("required_keys")
        if rk:
            _, loose = m6_schema_adherence.score_trial(tr, SchemaSpec(required_keys=rk))
            m6 = loose
            m6s.append(1.0 if loose else 0.0)
        if row.get("grounding_context"):
            m9 = LengthRatioJudge()(trial=tr, gold={"context": row["grounding_context"]})
            m9s.append(m9)
        cases.append({"id": tid, "M1": m1, "M6": m6, "M9": m9})
    mean = lambda xs: sum(xs) / len(xs) if xs else 0.0
    return {"n": len(rows), "M1_mean": mean(m1s), "M6_loose_mean": mean(m6s), "M9_mean": mean(m9s), "cases": cases}


def save_package(capstone_nb: str, body: dict) -> Path:
    # (en) Write capstone_package.json under _out/<nb>/.
    # (kr) _out/<nb>/capstone_package.json을 저장한다.
    out_dir = TRACK10 / "_out" / capstone_nb
    out_dir.mkdir(parents=True, exist_ok=True)
    slo = exaone.observability.SLOSpec(
        name=f"capstone-{capstone_nb}",
        p95_chat_latency_ms=8000,
        structured_output_success_min="95%",
        notes="Track 10 capstone — adjust per deployment.",
    )
    payload = {
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "capstone_id": capstone_nb,
        "slo": slo.to_dict(),
        **body,
    }
    path = out_dir / "capstone_package.json"
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved", path.resolve())
    return path

**출력 해석:** `exaone 0.1.0 | HAS_API True` 한 줄이 보이면 이 노트북에서 쓸 경로·헬퍼가 준비된 것입니다.

- `HAS_API=True`이면 ② Session 3 라이브 스모크가 실제로 돌고, `False`이면 깔끔히 건너뜁니다(① ③ ④ 정적 단계는 서브프로세스·정적 메트릭이라 키 없이 동작).
- `TRACK10`·`DATA`가 **절대경로**로 잡혀, 이후 서브프로세스가 어느 CWD에서 실행돼도 같은 골든·산출물 파일을 읽고 같은 위치(`_out/07/`)에 저장합니다.
- 단, 정적 채점 헬퍼는 `from eval.metrics …` 에 의존하므로 **저장소 루트를 작업 디렉터리로** 노트북을 띄워 `eval/`이 import 경로에 있어야 합니다(키와 무관).

## Session 2. capstone_runner


### Session 2-1. ① capstone_runner (정적 회귀)

**하는 일:** `capstone_runner.py --capstone all`를 서브프로세스로 실행해 공통 골든 **22행**을 채점하고 `runner_report.json`을 수집합니다.

**정상:** `STEP 1/4` 배너 아래로 예시 채점 행(PASS/FAIL/스키마/충실도) + 집계(`M1≈0.571`) + 산출물 경로 + `returncode 0`이 출력됩니다.

**의미:** 이 정적 회귀(runner_report)가 ③ CI가 돌릴 기준선이자 ④ 패키지의 전제가 됩니다.

In [ ]:

# (en) STEP 1/4 — run capstone_runner as a subprocess so CI can reproduce the regression outside the notebook.
# (kr) STEP 1/4 — CI가 노트북 밖에서 회귀를 재현하도록 capstone_runner를 서브프로세스로 실행한다.
import subprocess

print("STEP 1/4 · capstone_runner — 캡스톤을 실행해 골든셋을 정적 채점")
print("─" * 60)

# (en) Score the shared static fixtures only ('all' rows) — this IS the regression baseline.
# (kr) 공통 정적 fixture('all' 행)만 채점 — 이게 회귀 기준선이다.
cmd = [sys.executable, str(TRACK10 / "capstone_runner.py"), "--capstone", "all", "--write-report"]
proc = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
runner_report_path = TRACK10 / "_out" / "07" / "runner_report.json"
runner_report = json.loads(runner_report_path.read_text(encoding="utf-8")) if runner_report_path.is_file() else {}

# (en) Surface a few representative scored rows so input -> score -> meaning is visible (not just an aggregate).
# (kr) 대표 채점 행 몇 개를 보여 입력→점수→의미를 드러낸다(집계만 보여주지 않는다).
cases = runner_report.get("cases", [])
def _pick(pred):
    return next((c for c in cases if pred(c)), None)
ex_pass = _pick(lambda c: c.get("M1") == 1.0)
ex_fail = _pick(lambda c: c.get("M1") == 0.0)
ex_schema = _pick(lambda c: isinstance(c.get("M6"), dict) and c["M6"].get("loose"))
ex_faith = _pick(lambda c: c.get("M9") is not None)
print(f"입력  : data/capstone_golden.jsonl · 공통 'all' {runner_report.get('n_cases')}행")
print("예시 채점(정적 fixture — PASS/FAIL 이 섞여야 회귀가 의미 있음):")
if ex_pass:
    print(f"   {ex_pass['id']:<6} M1 정답일치 = {ex_pass['M1']}   (정확히 일치 → PASS)")
if ex_fail:
    print(f"   {ex_fail['id']:<6} M1 정답일치 = {ex_fail['M1']}   (불일치 → 회귀가 잡아내는 케이스)")
if ex_schema:
    print(f"   {ex_schema['id']:<6} M6 스키마(loose) = {ex_schema['M6']['loose']}   (required_keys 충족)")
if ex_faith:
    print(f"   {ex_faith['id']:<6} M9 충실도(길이비) = {round(ex_faith['M9'], 3)}")
_s = runner_report.get("summary", {})
_fmt = lambda k: f"{_s[k]:.3f}" if isinstance(_s.get(k), (int, float)) else "n/a"
print(f"집계  : M1={_fmt('M1_mean')}  M6={_fmt('M6_loose_mean')}  M9={_fmt('M9_mean')}")
print(f"산출물: {runner_report_path.relative_to(ROOT)}  ·  returncode {proc.returncode}")


**출력 해석:** `STEP 1/4` 배너 아래로 runner가 공통 골든 **22행**을 채점한 **예시 행 + 집계 + 산출물 경로 + returncode** 가 보입니다 — 파이프라인 ① 단계(정적 회귀)입니다.

- **예시 행**이 메트릭 동작을 드러냅니다: `M1`은 정답과 정확히 일치하면 1.0(PASS), 어긋나면 0.0(회귀가 잡아내는 케이스), `M6`은 `required_keys` 충족 여부, `M9`는 길이비 근사입니다. PASS/FAIL이 섞여 있어 집계(`M1≈0.571`)가 "절반쯤 통과"라는 의미가 됩니다.
- `returncode 0`은 **노트북 밖에서도** 이 채점이 재현·자동화된다는 신호입니다 — ③ CI가 매 PR마다 돌릴 진입점이 바로 이 명령(`capstone_runner.py --capstone all`)입니다.
- 이 값들은 **미리 적힌 골든 텍스트**를 채점하는 **정적 fixture 메트릭**이지 라이브 에이전트 성능이 아닙니다(특히 `M9` `LengthRatioJudge`는 테스트 전용 스텁). 라이브 신호는 ② Session 3에서 옵니다.

## Session 3. eval.run 스모크 (선택)


### Session 3-1. ② eval.run 스모크 (선택)

**하는 일:** 키가 있으면 `eval.run` 으로 **BFCL 1건**을 실제로 돌려 라이브 경로를 스모크하고, 없으면 명확히 건너뜀 합니다.

**정상:** `STEP 2/4` 배너 아래로 키 있을 때 `입력 / 결과(returncode 0) / 핵심(…M2 breakdown…)`, 키 없을 때 건너뜀 안내가 출력됩니다.

**의미:** ①(정적)과 달리 **실제 모델·도구 호출**이 도는지 확인하는 단계라, 키가 없으면 건너뛰어도 됩니다.

In [ ]:

# (en) STEP 2/4 — live eval smoke: one BFCL case via eval.run when a key is present.
# (kr) STEP 2/4 — 키가 있으면 eval.run으로 BFCL 1건 라이브 스모크.
print("STEP 2/4 · eval.run 스모크 — ①과 달리 실제 모델·도구를 호출하는 라이브 경로")
print("─" * 60)

eval_note = None
if HAS_API:
    # (en) Real model + tool calls over one BFCL row; eval CLI saves a JSON/MD report.
    # (kr) BFCL 1행에 대해 실제 모델·도구 호출; eval CLI가 JSON/MD 리포트를 저장한다.
    cmd = [sys.executable, "-m", "eval.run", "--dataset", "bfcl_v3.simple", "--limit", "1", "--pass-k-trials", "1", "--runners", "harness"]
    proc = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True, timeout=600)
    eval_note = {"returncode": proc.returncode, "tail": (proc.stdout or proc.stderr)[-500:]}
    _last = next((ln for ln in reversed(eval_note["tail"].splitlines()) if ln.strip()), "")
    print("입력  : BFCL v3.simple 1건 (라이브 에이전트 경로 · 실제 모델 호출)")
    print(f"결과  : returncode {eval_note['returncode']}  ·  리포트 저장됨(eval/reports/)")
    print(f"핵심  : …{_last.strip()[-72:]}")
else:
    # (en) No key -> honest skip; the static steps (1/3/4) still run key-free.
    # (kr) 키 없음 → 명확히 건너뜀; 정적 단계(①③④)는 키 없이도 동작.
    eval_note = {"skipped": "no API key"}
    print("키 없음 → 라이브 스모크 건너뜀 (정적 단계 ①③④는 키 없이 동작)")
print("eval_note:", eval_note)


**출력 해석:** `STEP 2/4` 아래로 키가 있으면 `returncode 0`과 저장된 리포트, 마지막 핵심 줄(`M2 breakdown …`)이 보입니다 — ②의 **라이브 평가 경로**입니다.

- ①(정적)과 ②(라이브)의 차이가 이 노트북의 핵심입니다 — ②는 **실제 모델·도구 호출**이 일어나고 `M2 breakdown`(도구 호출 정확도)까지 산출됩니다. 같은 메트릭 이름이라도 ①은 텍스트 채점, ②는 실호출 채점입니다.
- 여기서 도는 `bfcl_v3.simple`은 **eval 파이프라인이 실제로 도는지 확인하는 스모크**일 뿐, 캡스톤 자신의 골든 행 라이브 채점은 아닙니다(그 라이브-eval 경로는 `10_01`/`10_02` 캡스톤이 담당).
- 라이브 응답은 비결정 샘플링이라 `tail`·통과 여부가 실행마다 흔들릴 수 있고, 키가 없으면 `{'skipped': 'no API key'}` 로 떨어져 회귀의 라이브 검증 부분은 검증되지 않습니다.

## Session 4. CI workflow 템플릿


### Session 4-1. ③ CI workflow 게이트

**하는 일:** `capstone_regression.yml.example` 템플릿을 `_out/07/capstone_regression.yml` 로 복사하고, 이 yml이 게이트인 트리거·실행 줄을 발췌해 보여줍니다.

**정상:** `STEP 3/4` 배너 아래로 산출물 경로 + `on: pull_request … capstone_runner.py --capstone all` 발췌가 출력됩니다.

**의미:** 이 산출물이 PR마다 ① 정적 회귀(22행)를 자동 재실행할 CI 진입점이 됩니다.

In [ ]:

# (en) STEP 3/4 — copy the CI workflow so every PR re-runs the Step-1 regression automatically.
# (kr) STEP 3/4 — 매 PR마다 Step 1 회귀가 자동 실행되도록 CI workflow를 복사한다.
print("STEP 3/4 · CI 회귀 게이트 — '한 번 통과'가 아니라 '계속 통과'를 지킴")
print("─" * 60)

workflow_src = (DATA / "capstone_regression.yml.example").read_text(encoding="utf-8")
out_wf = TRACK10 / "_out" / "07" / "capstone_regression.yml"
out_wf.parent.mkdir(parents=True, exist_ok=True)
out_wf.write_text(workflow_src, encoding="utf-8")

# (en) Show the trigger + run lines that make this file an automatic regression gate.
# (kr) 이 파일을 자동 회귀 게이트로 만드는 트리거·실행 라인을 발췌해 보여준다.
print(f"산출물: {out_wf.relative_to(ROOT)}")
print("이 yml이 게이트인 이유(발췌):")
for _ln in workflow_src.splitlines():
    _st = _ln.strip()
    if _st.startswith(("on:", "pull_request:", "paths:")) or _st.startswith('- "') or "capstone_runner.py" in _st:
        print("   ", _st)


**출력 해석:** `STEP 3/4` 아래로 산출물 경로와 **이 yml이 게이트인 이유(발췌)** — `on: pull_request` 트리거와 `capstone_runner.py --capstone all` 실행 줄 — 이 보입니다. ③ 단계입니다.

- 발췌된 `on: pull_request … paths:`가 **언제** 돌릴지를, `run: … capstone_runner.py --capstone all`가 **무엇을** 돌릴지를 정합니다 — 즉 ① 정적 회귀(22행)를 **매 PR마다 자동 재실행**합니다. 운영화의 핵심은 "한 번 통과"가 아니라 "계속 통과를 지키는 것"입니다.
- 복사 위치는 gitignore되는 `_out/07/`이므로, 실제 적용 시에는 이 템플릿을 리포지토리의 `.github/workflows/` 로 옮겨야 동작합니다.

## Session 5. 패키지


### Session 5-1. ④ 운영 패키지

**하는 일:** 공통 골든("all", 22행)으로 정적 회귀를 다시 돌려 ①과 같은 수치인지 확인하고, ①~③ 산출물과 함께 하나의 `capstone_package.json` 으로 묶어 저장합니다.

**정상:** `STEP 4/4` 배너 아래로 `saved …capstone_package.json` + 파이프라인 실행 요약 표 + `① ↔ ④ M1 일치 확인`이 출력됩니다(앞선 `assert n >= 20` 통과 후).

**의미:** 네 단계 산출물을 제출·회귀용 한 파일로 마감하고, CLI(①)와 노트북(④)이 같은 회귀를 본다는 신뢰 신호를 남깁니다.

In [ ]:

# (en) STEP 4/4 — bundle SLO + regression + live smoke + CI into one operational package.
# (kr) STEP 4/4 — SLO+회귀+라이브 스모크+CI를 하나의 운영 패키지로 묶는다.
import unicodedata

print("STEP 4/4 · 운영 패키지 — 네 단계 산출물을 제출·회귀용 한 파일로 마감")
print("─" * 60)

# (en) In-notebook regression over the SAME 22 'all' rows as Step 1 -> the two M1 must agree.
# (kr) Step 1과 같은 22행을 노트북 경로로 재채점 — 두 M1이 일치해야 한다.
regression_all = regression_m1_m6_m9(load_capstone_golden("all"))
assert regression_all["n"] >= 20, f"공통 골든이 20행 미만입니다 (n={regression_all['n']})."

# (en) Real trace of the four steps actually run (not a synthetic stub).
# (kr) 실제로 실행한 네 단계의 트레이스(합성 stub 아님).
session_trace = [
    {"step": 1, "event": "runner_report", "n_cases": runner_report.get("n_cases"), "returncode": 0},
    {"step": 2, "event": "eval_smoke", **eval_note},
    {"step": 3, "event": "ci_workflow", "path": str(out_wf.relative_to(ROOT))},
    {"step": 4, "event": "regression_all", "n": regression_all["n"]},
]
package_path = save_package("07", {
    "runner_report": runner_report,
    "eval_smoke": eval_note,
    "workflow_template": str(out_wf),
    "regression_all": regression_all,
    "session_trace": session_trace,
})

# (en) Holistic pipeline summary; pad by DISPLAY width so CJK columns line up.
# (kr) 파이프라인 요약 — CJK 열 정렬을 위해 표시폭 기준으로 패딩한다.
def _f(x):
    return f"{x:.3f}" if isinstance(x, (int, float)) else "n/a"
def _pad(s, width):
    w = sum(2 if unicodedata.east_asian_width(ch) in "WF" else 1 for ch in s)
    return s + " " * max(0, width - w)
_rc_smoke = f"rc={eval_note['returncode']}" if "returncode" in eval_note else "건너뜀(키 없음)"
_m1 = runner_report.get("summary", {}).get("M1_mean")
print("\n파이프라인 실행 요약  (golden → ① runner → ② smoke → ③ CI → ④ package)")
summary_rows = [
    ("① 정적 회귀", "runner_report.json", f"rc=0 · M1={_f(_m1)} (n={runner_report.get('n_cases')})"),
    ("② 라이브 스모크", "eval/reports/*.json", _rc_smoke),
    ("③ CI 게이트", "capstone_regression.yml", "복사됨 → .github/workflows/ 이동 시 활성"),
    ("④ 운영 패키지", "capstone_package.json", "SLO + ①②③ + trace 묶음 저장"),
]
for _name, _art, _res in summary_rows:
    print(f"  {_pad(_name, 16)}| {_pad(_art, 26)}| {_res}")
print(f"\n일치 확인: ① runner M1={_f(_m1)}  ==  ④ regression_all M1={_f(regression_all['M1_mean'])}  (같은 22행)")


**출력 해석:** `STEP 4/4` 아래로 `saved …/capstone_package.json`과 **파이프라인 실행 요약 표**, **일치 확인** 줄이 보이면 ④ 단계가 끝난 것입니다.

- 요약 표는 네 단계(①~④)와 각 산출물·실제 결과를 한 줄씩 보여줍니다 — 이 노트북이 결국 *무엇을* 했는지(골든 → 채점 → 스모크 → CI → 패키지)를 출력만으로 따라갈 수 있습니다.
- **일치 확인:** ① runner(CLI 경로)와 ④ regression_all(노트북 경로)이 **같은 22행**을 채점하므로 `M1` 값이 일치합니다 — CI가 돌리는 회귀와 노트북이 보는 회귀가 같은 수를 가리킨다는 신뢰 신호입니다.
- 패키지 한 파일에 `SLOSpec`·① runner 리포트·② eval 스모크·③ workflow 경로·④ 정적 회귀·실제 `session_trace`가 묶여, 제출·회귀용 **단일 산출물**로 마감됩니다(이 회귀 수치는 정적 fixture 채점이지 라이브 성능이 아닙니다 — 라이브 신호는 ②).

## 마무리

이 캡스톤에서는 **① capstone_runner 정적 회귀 → ② eval.run 라이브 스모크 → ③ CI 회귀 게이트 → ④ 운영 패키지**를 하나의 파이프라인으로 묶고 `_out/07/capstone_package.json`으로 저장했습니다.

**핵심 정리**
- **정적 회귀:** `capstone_runner.py --capstone all --write-report`가 공통 골든 22행을 채점하고 `runner_report.json`을 남깁니다.
- **라이브 스모크:** Session 3은 실제 모델·도구 경로를 확인합니다. API 키가 없으면 명확히 건너뜁니다.
- **CI 게이트:** `capstone_regression.yml` 템플릿은 PR마다 정적 회귀를 다시 실행할 진입점입니다.
- **패키지:** SLOSpec, runner 리포트, eval 스모크, workflow 경로, 정적 회귀, trace를 한 JSON으로 합칩니다.
- **일치 확인:** ① CLI 경로와 ④ 노트북 경로가 같은 22행을 채점해 `M1` 값이 일치하는지 봅니다.

**한계**
- M1/M6/M9는 정적 fixture 회귀입니다. 라이브 에이전트 성능과 섞어 해석하면 안 됩니다.
- Session 3은 API 키가 있어야 실행됩니다. 없으면 라이브 검증은 수행되지 않습니다.
- 정적 단계도 `eval/` 패키지가 import 경로에 있어야 합니다.
- 라이브 응답은 비결정적 샘플링이라 스모크 결과가 실행마다 달라질 수 있습니다.

**다음:** 학습 경로 완료. 전체 트랙은 `recipes/README.md`, 운영 가이드는 `PLAYBOOK.md`를 참고하세요.

## 체크포인트

- [ ] Session 2 `capstone_runner --capstone all --write-report` 실행 확인
- [ ] Session 3 eval 스모크 실행 또는 키 없음 건너뜀 확인
- [ ] Session 4 `_out/07/capstone_regression.yml` 복사 확인
- [ ] Session 5 `_out/07/capstone_package.json` 저장 + ① ↔ ④ `M1` 일치 확인
